In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display


import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc

from test_code.dfield_approach_elastix.approach_images_similar.similarity_tables_reader import SimilarityTablesReader

from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from scipy.sparse.linalg import eigsh


## Read similarity table

In [6]:
matching_ids = pil.get_matching_ids()
sim_reader = SimilarityTablesReader(matching_ids)
sim_np = sim_reader.similarity_np

# np.fill_diagonal(sim_np, 1)
# sim_table_np = 1 / sim_table_np
# sim_table_np = 1 - sim_table_np
print(sim_np.shape)

print(f"min: {np.min(sim_np)}\t max: {np.max(sim_np)}\t mean: {np.mean(sim_np)}")

print(sim_np[:5, :5, 2])

(308, 308, 102)
min: 0.0	 max: 1.0	 mean: 0.2733301940224156
[[0.         0.772277   0.73946001 0.83538872 0.81761619]
 [0.772277   0.         0.77801205 0.78003491 0.79481651]
 [0.73946001 0.77801205 0.         0.7435273  0.77808193]
 [0.83538872 0.78003491 0.7435273  0.         0.81797121]
 [0.81761619 0.79481651 0.77808193 0.81797121 0.        ]]


## Find best clustering

In [11]:


def spectral_clustering_with_multi_metrics(similarity_matrix, num_clusters):
    # Obtener la matriz de afinidad sumando las matrices de similitud para cada métrica
    affinity_matrix = np.sum(similarity_matrix, axis=2)
    
    # Calcular la matriz laplaciana normalizada
    degree_matrix = np.diag(np.sum(affinity_matrix, axis=1))
    laplacian_matrix = degree_matrix - affinity_matrix
    normalized_laplacian = np.dot(np.linalg.inv(degree_matrix), laplacian_matrix)
    
    # Calcular los autovectores correspondientes a los autovalores más grandes
    num_eigenvectors = num_clusters + 1  # Se agrega uno para excluir el autovector constante
    eigenvalues, eigenvectors = eigsh(normalized_laplacian, k=num_eigenvectors, which='SM')
    
    # Clustering de los autovectores utilizando KMeans
    kmeans = KMeans(n_clusters=num_clusters)
    kmeans.fit(eigenvectors[:, 1:])  # Se excluye el autovector constante
    
    # Devolver las etiquetas de los clusters para cada elemento
    return kmeans.labels_

labels = spectral_clustering_with_multi_metrics(sim_np[:,:,2:5], 2)
print(labels)

[0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 0 0 1 1 1 0 1 1 1 0 0 0 0 0 0
 1 0 0 1 0 0 0 0 1 1 1 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0 0 0 0
 1 0 0 1 1 1 0 1 0 1 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 0 1 1 1 0 0 0 0 1 0 1 0
 0 0 1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 1 1 0 0 1 0
 0 1 0 1 1 1 1 0 0 1 0 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0
 0 0 0 0 1 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 1 0 0 1
 0 1 0 1 1 0 0 0 1 1 0 1 0 1 1 1 1 1 0 1 1 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1
 0 0 0 0 0 0 1 1 0 0 0 1 0 0 1 1 0 1 1 1 0 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 0
 0 1 1 0 1 1 0 1 0 1 1 1]
